In [0]:
# --- SETUP DO DASHBOARD ---
# Configuração de acesso (Igual aos notebooks anteriores)
storage_account_name = "f1datalakecarol2026"
storage_account_key = "INSIRA_AQUI_SUA_CHAVE_DE_ACESSO" 

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

container_gold = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("🚀 Carregando tabelas para o SQL...")

# Lendo os arquivos Delta e registrando como Views Temporárias
spark.read.format("delta").load(f"{container_gold}/fact_results").createOrReplaceTempView("fact_results")
spark.read.format("delta").load(f"{container_gold}/dim_drivers").createOrReplaceTempView("dim_drivers")
spark.read.format("delta").load(f"{container_gold}/dim_constructors").createOrReplaceTempView("dim_constructors")
spark.read.format("delta").load(f"{container_gold}/dim_races").createOrReplaceTempView("dim_races")

print("✅ Tabelas registradas! Agora o %sql vai funcionar.")

🚀 Carregando tabelas para o SQL...
✅ Tabelas registradas! Agora o %sql vai funcionar.


In [0]:
%sql
SELECT 
    d.driver_name, 
    SUM(f.points) as total_pontos
FROM fact_results f
JOIN dim_drivers d ON f.driver_id = d.driver_id
GROUP BY d.driver_name
ORDER BY total_pontos DESC
LIMIT 10

driver_name,total_pontos
Max,102.0
Lando,80.0
Sergio,79.0
Charles,71.0
Carlos,65.0
Oscar,36.0
George,32.0
Fernando,31.0
Lewis,12.0
Lance,9.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    c.team_name, 
    SUM(f.points) as total_pontos
FROM fact_results f
JOIN dim_constructors c ON f.team_id = c.team_id
GROUP BY c.team_name
HAVING total_pontos > 0

team_name,total_pontos
Mercedes,44.0
McLaren,116.0
Haas F1 Team,5.0
Ferrari,142.0
RB F1 Team,7.0
Red Bull,181.0
Aston Martin,40.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    ROUND((SUM(CASE WHEN f.status LIKE 'Finished' THEN 0 ELSE 1 END) / COUNT(*)) * 100, 2) AS taxa_churn
FROM fact_results f

taxa_churn
42.0


Databricks visualization. Run in Databricks to view.